In [1]:
#import require lib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
data=pd.read_csv(r"C:\Users\ARUN\Desktop\orders.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Order Id          9994 non-null   int64 
 1   Order Date        9994 non-null   object
 2   Ship Mode         9993 non-null   object
 3   Segment           9994 non-null   object
 4   Country           9994 non-null   object
 5   City              9994 non-null   object
 6   State             9994 non-null   object
 7   Postal Code       9994 non-null   int64 
 8   Region            9994 non-null   object
 9   Category          9994 non-null   object
 10  Sub Category      9994 non-null   object
 11  Product Id        9994 non-null   object
 12  cost price        9994 non-null   int64 
 13  List Price        9994 non-null   int64 
 14  Quantity          9994 non-null   int64 
 15  Discount Percent  9994 non-null   int64 
dtypes: int64(6), object(10)
memory usage: 1.2+ MB


In [3]:
data.columns=data.columns.str.lower().str.replace(" ","_")
data.head()

,order_id,order_date,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,product_id,cost_price,list_price,quantity,discount_percent
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,240,260,2,2
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,600,730,3,3
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,10,10,2,5
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,780,960,5,2
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,20,20,2,5


In [4]:
data['order_date']=pd.to_datetime(data['order_date'])
data['order_year']=data['order_date'].dt.year
data['order_month']=data['order_date'].dt.month

In [5]:
data['discount_amount'] = (data['list_price'] * data['discount_percent']) / 100

# price after discount (per unit)
data['sales_price'] = data['list_price'] - data['discount_amount']

# total revenue
data['revenue'] = data['sales_price'] * data['quantity']

# total cost
data['total_cost'] = data['cost_price'] * data['quantity']

# profit
data['profit'] = round(data['revenue'] - data['total_cost'], 2)

In [6]:
data.columns


Index(['order_id', 'order_date', 'ship_mode', 'segment', 'country', 'city',
       'state', 'postal_code', 'region', 'category', 'sub_category',
       'product_id', 'cost_price', 'list_price', 'quantity',
       'discount_percent', 'order_year', 'order_month', 'discount_amount',
       'sales_price', 'revenue', 'total_cost', 'profit'],
      dtype='object')

In [7]:
data.isnull().sum()
data[data.isnull().any(axis=1)]
data['ship_mode'].fillna('unknown',inplace=True)

C:\Users\ARUN\AppData\Local\Temp\ipykernel_1592\109220146.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['ship_mode'].fillna('unknown',inplace=True)


In [8]:
data.isnull().sum()

order_id            0
order_date          0
ship_mode           0
segment             0
country             0
city                0
state               0
postal_code         0
region              0
category            0
sub_category        0
product_id          0
cost_price          0
list_price          0
quantity            0
discount_percent    0
order_year          0
order_month         0
discount_amount     0
sales_price         0
revenue             0
total_cost          0
profit              0
dtype: int64

In [9]:
data.duplicated().sum()

np.int64(0)

In [10]:
print(f"One null value handled and data has 0 duplicate values . shape of file is, {data.shape}")

One null value handled and data has 0 duplicate values . shape of file is, (9994, 23)


In [11]:
!pip install sqlalchemy

In [12]:
import sqlalchemy 
from sqlalchemy import create_engine, text
engine = create_engine( "mssql+pyodbc://Agamagizhan/retail_order" "?driver=ODBC+Driver+17+for+SQL+Server" "&trusted_connection=yes" )
data.to_sql("retail_order", engine, if_exists="replace", index=False) 
print("✅ Uploaded!", len(data), "rows")

✅ Uploaded! 9994 rows


In [13]:
!pip install pyodbc

In [14]:
import pyodbc
conn=pyodbc.connect("DRIVER={ODBC Driver 17 for SQL Server};" 
"SERVER=Agamagizhan;"
"DATABASE=retail_order;"
"Trusted_Connection=yes;")
cursor=conn.cursor()
cursor.execute("select * from retail_order")
for row in cursor:
    print(row)

(1, datetime.datetime(2023, 3, 1, 0, 0), 'Second Class', 'Consumer', 'United States', 'Henderson', 'Kentucky', 42420, 'South', 'Furniture', 'Bookcases', 'FUR-BO-10001798', 240, 260, 2, 2, 2023, 3, 5.2, 254.8, 509.6, 480, 29.6)
(2, datetime.datetime(2023, 8, 15, 0, 0), 'Second Class', 'Consumer', 'United States', 'Henderson', 'Kentucky', 42420, 'South', 'Furniture', 'Chairs', 'FUR-CH-10000454', 600, 730, 3, 3, 2023, 8, 21.9, 708.1, 2124.3, 1800, 324.3)
(3, datetime.datetime(2023, 1, 10, 0, 0), 'Second Class', 'Corporate', 'United States', 'Los Angeles', 'California', 90036, 'West', 'Office Supplies', 'Labels', 'OFF-LA-10000240', 10, 10, 2, 5, 2023, 1, 0.5, 9.5, 19.0, 20, -1.0)
(4, datetime.datetime(2022, 6, 18, 0, 0), 'Standard Class', 'Consumer', 'United States', 'Fort Lauderdale', 'Florida', 33311, 'South', 'Furniture', 'Tables', 'FUR-TA-10000577', 780, 960, 5, 2, 2022, 6, 19.2, 940.8, 4704.0, 3900, 804.0)
(5, datetime.datetime(2022, 7, 13, 0, 0), 'Standard Class', 'Consumer', 'United